In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from torchvision import datasets, transforms
import torch
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import accuracy_score
import pandas as pd
from bertopic import BERTopic
from sklearn.pipeline import Pipeline
from scipy.sparse import csr_matrix
from sklearn.model_selection import GridSearchCV
from bertopic.vectorizers import ClassTfidfTransformer
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import math
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import Dataset
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
transform = transforms.Compose([ #normalized data, can be done without normalized data but threshold will need to be changed
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [26]:

class MaxPooledMNIST(Dataset):
    def __init__(self, dataset, kernel_size=2, stride=2):
        self.dataset = dataset
        self.max_pool = nn.AvgPool2d(kernel_size=kernel_size, stride=stride)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]  # Get image and label
        with torch.no_grad():
            pooled_image = self.max_pool(image)  # Apply max pooling
        return pooled_image, label

    @property
    def targets(self):  # Keep labels accessible like datasets.MNIST
        return self.dataset.targets
def encode_images(region_size, overlap, threshold, dataset):
    def pad_image(image, region_size, overlap):
        height, width = image.shape[1], image.shape[2]
        stride = region_size - overlap
        pad_height = (stride - (height % stride)) % stride
        pad_width = (stride - (width % stride)) % stride
        padded_image = torch.nn.functional.pad(image, (0, pad_width, 0, pad_height), mode='constant', value=0)
        return padded_image

    def regions(image, size, overlap):
        padded_image = pad_image(image, size, overlap)
        stride = size - overlap
        regions = []
        for i in range(0, padded_image.shape[1] - overlap, stride):
            for j in range(0, padded_image.shape[2] - overlap, stride):
                region = padded_image[0, i:i+size, j:j+size]
                regions.append(region)
        return regions

    def regions(image, size, overlap):
        padded_image = pad_image(image, size, overlap)
        regions = []
        for i in range(0, padded_image.shape[1], size):
            for j in range(0, padded_image.shape[2], size):
                region = padded_image[0, i:i+size, j:j+size]
                regions.append(region)
        return regions

    def encode(region, threshold):
        region = torch.where(region < threshold, 0, 1)
        binary_str = ''.join(map(str, region.flatten().int().tolist()))
        return int(binary_str, 2)

    encoded_images = []
    for image, label in dataset:
        r = regions(image, region_size, overlap)
        r = [encode(region, threshold) for region in r]
        encoded_images.append([x for x in r if x != 0]) 

    return encoded_images

def create_groupeddf(encoded_images, dataset):
    text_data = [' '.join(map(str, img)) for img in encoded_images]
    df = pd.DataFrame({'Document': text_data, 'Label': dataset.targets.tolist()})
    return df.groupby('Label', as_index=False).agg({'Document': ' '.join})

def extract_ctfidf_features(groupeddf, score_threshold):
    ctfidf, features = BERTopic()._c_tf_idf(groupeddf, fit=True)
    ctfidf_array = ctfidf.toarray()

    ctfidf_features = {}
    for idx, topic in enumerate(groupeddf['Label']):
        top_indices = [i for i in range(len(features)) if ctfidf_array[idx][i] >= score_threshold]
        scaled_features = []
        for i in top_indices:
            term = features[i]
            count = max(1, int(ctfidf_array[idx][i] * 20000))
            scaled_features.extend([term] * count)
        ctfidf_features[topic] = scaled_features

    return ctfidf_features

def max_pool(encoded_regions, pool_size):
    pooled_output = []
    
    for i in range(0, len(encoded_regions), pool_size):
        pooled_output.append(max(encoded_regions[i:i+pool_size]))
    
    return pooled_output

def max_pool_ctfidf(ctfidf_features, pool_size):
    return {key: max_pool(value, pool_size) for key, value in ctfidf_features.items()}

def model_with_params(region_size, overlap, threshold, score_threshold, pool_size, train_dataset, test_dataset):
    train_dataset = MaxPooledMNIST(train_dataset)
    encoded_train = encode_images(region_size, overlap, threshold, train_dataset)
    groupeddf = create_groupeddf(encoded_train, train_dataset)
    ctfidf_features = extract_ctfidf_features(groupeddf, score_threshold)
    #ctfidf_features = max_pool_ctfidf(ctfidf_features, pool_size)
    X_train = [' '.join(words) for words in ctfidf_features.values()]
    y_train = list(ctfidf_features.keys())

    X_test = [' '.join(map(str, img)) for img in encode_images(region_size, overlap, threshold, test_dataset)]
    y_test = test_dataset.targets.tolist()

    vectorizer = CountVectorizer()
    X_train_vectors = vectorizer.fit_transform(X_train)
    X_test_vectors = vectorizer.transform(X_test)

    model = MultinomialNB()
    model.fit(X_train_vectors, y_train)
    y_pred = model.predict(X_test_vectors)

    return accuracy_score(y_test, y_pred), groupeddf, ctfidf_features

score, gdf, ctfidf_features = model_with_params(6, 2, -.95, .00025, 0, train_dataset, test_dataset)
print(score*100)

# for i in range(1, 101, 10):
#     score, gdf, ctfidf_features = model_with_params(6, 2, -.95, .00025, i, train_dataset, test_dataset)
#     print(i, " Pool: ", score)

11.17
